# Group 3 — Project 2 (Delsys + force plates)

This notebook shows how to load your **Delsys** and **force-plate** data, **synchronize** them, and read off aligned signals. There are **two takes**: `001` and `003` (Delsys `Trial_1` ↔ take `001`, `Trial_2` ↔ take `003`).

**Sensors** — Delsys `Ch1–4` = EMG (+accel), `Ch12` = EKG, `Ch13` = **analog SYNC** (Motive record gate). Force plates come **through Motive**, so they're on the **Motive clock**; the Delsys is on its own clock.

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data (take 001)

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
def get(p): return hf_hub_download(REPO, p, repo_type="dataset")
delsys_csv = get("group_3_2/delsys/Trial_1.csv")                              # Trial_2 = take 003
fp1_csv    = get("group_3_2/bertec/group3_project2_001_forceplate_1.csv")
fp2_csv    = get("group_3_2/bertec/group3_project2_001_forceplate_2.csv")
print("downloaded")

### 2 · Load each modality (raw, on its own clock)
**Delsys** — EMG (and EKG on Ch12):

In [ ]:
import delsys, numpy as np, matplotlib.pyplot as plt
lf = delsys.Log(delsys_csv)
print("sensors:", lf.sensor_numbers)
emg1 = lf.find(sensor_number=1, as_="sensor")[0].emg
plt.figure(figsize=(12,2.5)); plt.plot(emg1.t, emg1()); plt.xlim(30,35); plt.title("EMG sensor 1 (Delsys clock)"); plt.show()

**Force plates** — `MotiveLog` (pasted; not on pip). `MocapTime` is the Motive clock.

In [ ]:
import csv as _csv, numpy as np, pandas as pd, pysampled
class MotiveLog:
    """Load a Motive force-plate 'Device export' CSV. MocapTime is the OptiTrack clock."""
    def __init__(self, fname):
        hdr = {}
        with open(fname, newline='') as f:
            for rc, row in enumerate(_csv.reader(f)):
                if not row: continue
                if row[0] == 'MocapFrame': break
                v = row[1] if len(row) > 1 else None
                try: hdr[row[0]] = float(v)
                except (TypeError, ValueError): hdr[row[0]] = v
        self.data = pd.read_csv(fname, skiprows=rc).rename(columns=lambda x: x.strip())
        self.sr = hdr['Mocap Rate'] * hdr.get('Mocap Rate Multiple', 1)
    def __getattr__(self, k):
        if k in ('Fx','Fy','Fz','Mx','My','Mz','Cx','Cy','Cz','Tz'):
            return pysampled.Data(self.data[k].values, sr=self.sr)
        raise AttributeError(k)

In [ ]:
fp1 = MotiveLog(fp1_csv); fp2 = MotiveLog(fp2_csv)
print("force-plate sr:", fp1.sr, "Hz")
fz1 = np.abs(np.asarray(fp1.Fz()).ravel()); fp_t = np.arange(len(fz1))/fp1.sr
plt.figure(figsize=(12,2.5)); plt.plot(fp_t, fz1); plt.xlim(30,35); plt.title("force plate 1 |Fz| (Motive clock)"); plt.show()

### 3 · Synchronize — the key step
The **force plates** are on the **Motive clock** (start at 0); the **Delsys** is on its own clock. The Delsys recorded Motive's **record gate** on an analog channel — its rising edge is the offset between the clocks.

In [ ]:
import numpy as np
def motive_gate_offset(lf):
    """Delsys analog sensor 13, channel 0 = Motive's record gate (HIGH while Motive records).
    Its rising edge is the Delsys-clock time at which Motive (force plates + markers) began at t=0."""
    a = lf.find(sensor_number=13, as_="sensor")[0].analog
    g = np.asarray(a())[:, 0]; t = np.asarray(a.t)
    return float(t[np.argmax(g > 0.5*(g.min()+g.max()))])

In [ ]:
offset = motive_gate_offset(lf)
print(f"Motive started at Delsys t = {offset:.2f} s")
# move ANY Delsys signal onto the Motive clock:   t_motive = t_delsys - offset

### 4 · Load synchronized signals

In [ ]:
env = emg1.bandpass(20,450).envelope()
emg_t = np.asarray(env.t) - offset
fig, ax = plt.subplots(2,1, figsize=(13,5), sharex=True)
ax[0].plot(fp_t, fz1); ax[0].set_ylabel("|Fz| (N)"); ax[0].set_title("force plate (Motive clock)")
ax[1].plot(emg_t, np.asarray(env()).ravel(), color="tab:orange"); ax[1].set_ylabel("EMG env")
ax[1].set_xlabel("Motive time (s)"); ax[1].set_title("Delsys EMG shifted onto the Motive clock — now aligned")
[a.set_xlim(20,50) for a in ax]; plt.tight_layout(); plt.show()

### 5 · Your turn
- Everything is now on the **Motive clock** (force plate as-is; Delsys after `− offset`). Line up muscle activity with ground-reaction force in your task windows.
- Other sensors: EMG `sensor_number=2,3,4`; **EKG** on `sensor_number=12` (`.ekg.find_rpeaks()` for heart rate); `.acc` on each sensor for acceleration.
- **Take 003** is `Trial_2` + the `...003...` force plates — repeat with those to compare the two takes.